In [ ]:
import torch
import shutil
import json
import yaml
import random
import cv2
import re, random
import matplotlib.pyplot as plt
from ultralytics import YOLO
from pathlib import Path
from collections import Counter

In [2]:
# 디바이스 설정
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.xpu.is_available():
    DEVICE = torch.device('xpu')
else:
    DEVICE = torch.device('cpu')

print(DEVICE)

cuda


In [3]:
# 데이터 경로 설정
# 차량 파손 여부 /유형 탐지 목표라 damgage만 사용

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / 'data' / '자동차파손'
TRAIN_DIR = DATA_DIR / 'Training'
VAL_DIR = DATA_DIR / 'Validation'
YOLO_DIR = DATA_DIR / 'Car_Damaged_YOLO'

TRAIN_IMAGE = TRAIN_DIR / '1.원천데이터' / 'TS_damage' / 'damage'
TRAIN_LABEL = TRAIN_DIR / '2.라벨링데이터' / 'TL_damage' / 'damage'

VAL_IMAGE = VAL_DIR / '1.원천데이터' / 'VS_damage' / 'damage'
VAL_LABEL = VAL_DIR / '2.라벨링데이터' / 'VL_damage' / 'damage'

In [4]:
"""
{"info": {"name": "external", "date_created": "04/20/2022"},
 "images": {"id": 1, "width": 800, "height": 600, "file_name": "0000002_as-0036229.jpg"}, 
 "annotations": [{"id": 2, "image_id": 1, 
        "category_id": "as-0036229", 
        "segmentation": [
                [
                    [
                        [438, 504], [440, 443], [436, 419], [439, 415], [429, 382], [404, 351], [363, 304], [414, 351], [420, 333], [425, 316], [427, 312], [418, 296], [409, 280], [363, 229], [336, 206], [336, 162], [347, 159], [408, 209], [416, 216], [447, 224], [459, 233], [468, 229], [485, 222], [495, 219], [516, 216], [534, 216], [508, 237], [486, 259], [466, 277], [465, 289], [469, 345], [469, 371], [449, 348], [436, 336], [436, 369], [450, 404], [451, 423], [447, 485], [438, 504]
                    ]
                ]
            ], 
            "area": 14977.0, "bbox": [336, 159, 198, 345], 
        "damage": "Breakage", 
        "part": null, 
        "year": 2020, 
        "color": "Black", 
        "level": null, 
        "repair": ["Rear bumper:coating,exchange"]}], 
 "categories": {"id": "as-0036229", "supercategory_name": "Full-size car"}}
"""

"""
damage 라벨 유형
Scratched, Separated, Crushed, Breakage
"""

'\ndamage 라벨 유형\nScratched, Separated, Crushed, Breakage\n'

In [5]:
# 클래스: AI Hub 데이터 명세
# 파손, 찌그러짐, 스크래치, 이격
CLASS_NAMES = ['Breakage', 'Crushed', 'Scratched', 'Separated']
CLASS_MAP = {name: idx for idx, name in enumerate(CLASS_NAMES)}
print(CLASS_MAP)

{'Breakage': 0, 'Crushed': 1, 'Scratched': 2, 'Separated': 3}


In [ ]:
# COCO를 YOLO로 변환
# COCO : [x1,y1,x2,y2,x3,y3...]
# YOLO : [class_id,x1,y1,x2,y2,x3,y3...] : 정규화

def conv_coco_yolo(class_name,segmentation,img_width,img_height):
    class_id = CLASS_MAP[class_name]
    results = []

    for seg in segmentation:
        for polygon in seg:
            coord = []
            for x,y in polygon:
                coord.append(x / img_width)
                coord.append(y / img_height)

            results.append([class_id] + coord)

    return results


In [ ]:
files = list(VAL_IMAGE.glob('*.jpg'))
case_ids = sorted(set(re.match(r'\d+_(.+)\.jpg', f.name).group(1) for f in files))

random.seed(2026)
random.shuffle(case_ids)
half = len(case_ids) // 2
val_cases = set(case_ids[:half])
test_cases = set(case_ids[half:])

val_files = [f for f in files if re.match(r'\d+_(.+)\.jpg', f.name).group(1) in val_cases]
test_files = [f for f in files if re.match(r'\d+_(.+)\.jpg', f.name).group(1) in test_cases]

print(len(val_files), len(test_files))

24646 25799


In [ ]:
# 이미지 + 라벨을 YOLO 폴더 구조로 정리
# images/<split>/*.jpg, labels/<split>/*.txt

def build_yolo_dataset(image_files, label_dir, split):
    img_out_dir = YOLO_DIR / 'images' / split
    label_out_dir = YOLO_DIR / 'labels' / split
    img_out_dir.mkdir(parents=True, exist_ok=True)
    label_out_dir.mkdir(parents=True, exist_ok=True)

    skipped = 0
    for img_path in image_files:
        label_path = label_dir / f'{img_path.stem}.json'
        if not label_path.exists():
            skipped += 1
            continue

        with open(label_path, encoding='utf-8') as f:
            data = json.load(f)

        width = data['images']['width']
        height = data['images']['height']

        lines = []
        for ann in data['annotations']:
            damage = ann.get('damage')
            if damage is None:
                continue
            for row in conv_coco_yolo(damage, ann['segmentation'], width, height):
                lines.append(' '.join(map(str, row)))

        if not lines:
            skipped += 1
            continue

        shutil.copy2(img_path, img_out_dir / img_path.name)
        (label_out_dir / f'{img_path.stem}.txt').write_text('\n'.join(lines), encoding='utf-8')

    print(f'[{split}] {len(image_files) - skipped} / {len(image_files)} 처리 완료 (skip: {skipped})')

In [ ]:
train_files = list(TRAIN_IMAGE.glob('*.jpg'))

build_yolo_dataset(train_files, TRAIN_LABEL, 'train')
build_yolo_dataset(val_files, VAL_LABEL, 'val')
build_yolo_dataset(test_files, VAL_LABEL, 'test')

In [ ]:
# data.yaml 작성
yaml_data = {
    'path': str(YOLO_DIR),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': {i: name for i, name in enumerate(CLASS_NAMES)},
}

YAML_PATH = YOLO_DIR / 'Car_Damaged.yaml'

with YAML_PATH.open( 'w', encoding='utf-8') as f:
    yaml.safe_dump(yaml_data, f, allow_unicode=True, sort_keys=False)

print(YAML_PATH.read_text(encoding='utf-8'))

In [ ]:
model = YOLO("yolo11s-seg.pt")
print("Model task: ", model.task)

In [ ]:
train_results = model.train(
    data=str(yaml_data),
    epochs=5,
    imgsz=640,
    batch=4,
    device=DEVICE,
    workers=0,
    seed=2026,
    patience=20,
    name="starbucks_yolo11s_seg",
    exist_ok=True
)

In [ ]:
RUN_DIR = PROJECT_DIR / 'runs' / 'segment' / 'car_damaged_yolo11s_seg'
BEST_MODEL_PATH = RUN_DIR / 'weights' / 'best.pt'
print('best.pt: ', BEST_MODEL_PATH)

In [ ]:
best_model = YOLO(str(BEST_MODEL_PATH))
print("Task:",best_model.task)
print('Classes: ',best_model.names)

In [ ]:
metrics = best_model.val(
    data=str(YAML_PATH),
    split='val',
    imgsz=640,
    device=DEVICE
)

In [ ]:
print('Box mAP50: ', metrics.box.map50)
print('Box mAP50-95: ', metrics.box.map)
print('Mask mAP50: ', metrics.seg.map50)
print('Mask mAP50-95: ', metrics.seg.map5)

In [ ]:
TEST_IMAGE_DIR = YOLO_DIR / 'images' / 'test'
predict_results = best_model.predict(
    source=str(TEST_IMAGE_DIR),
    imgsz=640,
    conf=0.30,
    device=DEVICE,
    retina_masks=True,
    save=True,
    name='car_damaged_yolo11s_set_predict',
    exist_ok=True
)

In [ ]:
for result in predict_results[:3]:
    print("=" * 70)
    print("image:", result.path)

    if result.boxes is not None:
        print("classes    :", result.boxes.cls.cpu().tolist())
        print("confidence :", result.boxes.conf.cpu().tolist())
        print("boxes xyxy :")
        print(result.boxes.xyxy.cpu().numpy())

    if result.masks is not None:
        print("mask tensor shape:", tuple(result.masks.data.shape))
        print("polygon 개수     :", len(result.masks.xy))
    else:
        print("검출된 segmentation mask가 없습니다.")

In [ ]:
if not predict_results:
    raise RuntimeError("추론 결과가 없습니다.")

result = predict_results[0]

plotted_bgr = result.plot()
plotted_rgb = cv2.cvtColor(
    plotted_bgr,
    cv2.COLOR_BGR2RGB
)

plt.figure(figsize=(10, 8))
plt.imshow(plotted_rgb)
plt.axis("off")
plt.title("YOLO11 Car Damaged Instance Segmentation")
plt.show()